# FujiCV — Synthetic Multi-Label Classification on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dsabarinathan/fujicv/blob/main/examples/colab_multilabel.ipynb)

Trains **ResNet-18** on a **synthetic multi-label dataset** where each image can
simultaneously belong to multiple binary attribute classes.
No external downloads needed — images are generated in-notebook.
All results saved to `/content/results_multilabel/`.

> **Tip:** Set runtime to GPU → Runtime → Change runtime type → **T4 GPU**

## 1. Install FujiCV

In [ ]:
!pip install fujicv timm -q
print('Done.')

## 2. Imports & config

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader

import fujicv
from fujicv.engine.trainer import Trainer
from fujicv.eval.plots import plot_loss_curves
from fujicv.losses.classification import BCEWithLogitsLoss
from fujicv.metrics.multilabel import HammingScore, MeanAveragePrecision
from fujicv.models.builder import ModelBuilder
from fujicv.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

SEED        = 42
IMAGE_SIZE  = 32
BATCH_SIZE  = 64
EPOCHS      = 10
LR          = 1e-3
OUTPUT_DIR  = '/content/results_multilabel'
LABEL_NAMES = ['has_red', 'has_green', 'has_blue', 'is_bright', 'is_dark']
NUM_CLASSES = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)
device = fujicv.get_device()
print(f'Device: {device}')

## 3. Generate synthetic multi-label dataset

Each 32×32 RGB image gets 5 binary labels derived from actual pixel statistics:
- `has_red` / `has_green` / `has_blue` — channel mean > 128
- `is_bright` — overall mean > 128
- `is_dark`   — overall mean ≤ 128

Note: `is_bright` and `is_dark` are mutually exclusive by construction.

In [ ]:
np.random.seed(SEED)
rows = []
img_dir = '/content/synthetic_multilabel/images'
os.makedirs(img_dir, exist_ok=True)

for i in range(1000):
    arr = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
    labels = {
        'has_red':   int(arr[:, :, 0].mean() > 128),
        'has_green': int(arr[:, :, 1].mean() > 128),
        'has_blue':  int(arr[:, :, 2].mean() > 128),
        'is_bright': int(arr.mean() > 128),
        'is_dark':   int(arr.mean() <= 128),
    }
    fname = f'img_{i:04d}.jpg'
    Image.fromarray(arr).save(f'{img_dir}/{fname}')
    rows.append({'filename': fname, **labels})

df = pd.DataFrame(rows)
df.to_csv('/content/synthetic_multilabel/labels.csv', index=False)
print(f'Generated {len(df)} images.')
print('Label distribution:')
print(df[LABEL_NAMES].mean().round(3))
df.head()

## 4. Build DataLoaders via CSVImageDataset

In [ ]:
from fujicv.data.csv_dataset import CSVImageDataset
from fujicv.data.transforms import get_train_transforms, get_val_transforms
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

train_ds = CSVImageDataset(
    csv_path='/content/synthetic_multilabel/labels.csv',
    img_dir=img_dir,
    label_col=LABEL_NAMES,
    task='multilabel',
    transform=get_train_transforms(IMAGE_SIZE, level='light'),
    subset_df=train_df,
)
val_ds = CSVImageDataset(
    csv_path='/content/synthetic_multilabel/labels.csv',
    img_dir=img_dir,
    label_col=LABEL_NAMES,
    task='multilabel',
    transform=get_val_transforms(IMAGE_SIZE),
    subset_df=val_df,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

## 5. Build model

In [ ]:
model = ModelBuilder(
    backbone_name='resnet18',
    backbone_source='timm',
    pretrained=True,
    task='multilabel',
    num_outputs=NUM_CLASSES,
    image_size=IMAGE_SIZE,
).build()

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ResNet-18 multi-label | {total:.1f}M parameters | {NUM_CLASSES} output heads')

## 6. Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=BCEWithLogitsLoss(),
    metrics={'hamming': HammingScore(), 'mAP': MeanAveragePrecision()},
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    task='multilabel',
    output_dir=OUTPUT_DIR,
    monitor_metric='val_loss',
    mixed_precision=True,
    early_stopping_patience=5,
)

history = trainer.train()

## 7. Training curves

In [ ]:
fig = plot_loss_curves(history)
fig.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved loss curves.')

## 8. Per-label precision / recall

In [ ]:
from sklearn.metrics import classification_report as sk_report

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        logits = model(imgs.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_preds.append((probs > 0.5).astype(int))
        all_targets.append(labels.numpy().astype(int))

all_preds   = np.concatenate(all_preds,   axis=0)
all_targets = np.concatenate(all_targets, axis=0)

report = sk_report(all_targets, all_preds, target_names=LABEL_NAMES, zero_division=0)
print(report)
with open(f'{OUTPUT_DIR}/classification_report.txt', 'w') as f:
    f.write(report)

## 9. Summary

In [ ]:
best_loss    = min(history.metrics.get('val_loss',    [float('inf')]))
best_hamming = max(history.metrics.get('val_hamming', [0.0]))
best_map     = max(history.metrics.get('val_mAP',     [0.0]))

print(f"""
========================================
 FujiCV Multi-Label Classification
========================================
 Model      : ResNet-18 (pretrained)
 Labels     : {LABEL_NAMES}
 Epochs     : {EPOCHS}  |  Device: {device}
 Best Val BCE Loss     : {best_loss:.5f}
 Best Val Hamming Score: {best_hamming:.4f}
 Best Val mAP          : {best_map:.4f}
========================================
 Saved to /content/results_multilabel/
   loss_curves.png   classification_report.txt
   best.pt   history.csv
========================================
""")

for f in sorted(os.listdir(OUTPUT_DIR)):
    kb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<40} {kb:6.1f} KB')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/fujicv_multilabel_results', 'zip', OUTPUT_DIR)
files.download('/content/fujicv_multilabel_results.zip')